# Twilight Forecast History

**Authors:** Brian Brondel, B. Stalder

Queries the Rubin Observatory EFD for historical `WeatherForecast.hourlyTrend` temperature predictions and shows how successive forecasts estimated the temperature at the end of evening nautical twilight.

## Imports & Constants

Standard scientific Python stack plus Astropy for solar-position calculations, SciPy's Brent root-finder for precise twilight crossing times, and the LSST EFD client for telemetry queries.

Key constants:
- `DELTA_TIME` – 5-minute cadence of the `hourlyTrend` temperature grid.
- `NAUTICAL_TWILIGHT_DEG` / `ASTRONOMICAL_TWILIGHT_DEG` – sun-altitude thresholds (−12° and −18°) used as the forecast target time.
- `RUBIN` – ITRS location of the Vera C. Rubin Observatory on Cerro Pachón.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import AltAz, EarthLocation, get_sun
from astropy.time import Time, TimeDelta
from scipy.optimize import brentq

from lsst_efd_client import EfdClient

DELTA_TIME = 5 * 60  # seconds between hourlyTrend.temperature samples
ASTRONOMICAL_TWILIGHT_DEG = -18.0
NAUTICAL_TWILIGHT_DEG = -12.0
RUBIN = EarthLocation.of_site("Rubin")

## Configuration

Set the EFD instance and the query time window. `START`/`END` default to the past 8 hours relative to now; adjust them to inspect a specific night. Change `EFD_NAME` to `"summit_efd"` when running at the summit.

In [ ]:
# Query window (UTC). Adjust as needed.
END = Time.now()
START = END - TimeDelta(8 * 3600, format="sec")

# EFD instance: "summit_efd", "usdf_efd", "tucson_teststand_efd", etc.
EFD_NAME = "usdf_efd"
TOPIC = "lsst.sal.WeatherForecast.hourlyTrend"

## Connect to EFD & Discover Fields

Open a connection to the chosen EFD instance and enumerate all `temperature*` fields published in `hourlyTrend`. The fields are sorted by their integer suffix so they correspond to consecutive 5-minute forecast steps ahead of the publication timestamp.

In [ ]:
client = EfdClient(EFD_NAME)
fields = await client.get_fields(TOPIC)
temperature_fields = sorted(
    [
        f
        for f in fields
        if f.startswith("temperature") and f[len("temperature") :].isdigit()
    ],
    key=lambda f: int(f[len("temperature") :]),
)
print(f"Found {len(temperature_fields)} temperature grid points per sample.")

## Fetch Forecast Data

Query the EFD for all `hourlyTrend` messages published in the configured time window. Each row is one forecast issuance: `private_sndStamp` is the UTC Unix timestamp when the SAL publisher sent the message, and the `temperature0…temperatureN` columns contain the predicted temperatures at successive 5-minute steps ahead.

In [ ]:
df = await client.select_time_series(
    TOPIC,
    ["private_sndStamp"] + temperature_fields,
    START,
    END,
)
print(f"Fetched {len(df)} hourlyTrend samples.")
df.head()

## Helper Functions

Three utilities used in the computation step:

- **`sun_altitude_deg(t)`** – returns the sun's altitude above the horizon (degrees) at time `t` from Rubin's location, using pressure = 0 to skip atmospheric refraction.
- **`next_end_of_evening_twilight(after, target_alt)`** – scans forward hour-by-hour from `after` to bracket the next descending solar crossing of `target_alt`, then pinpoints it to sub-second precision with Brent's method.
- **`interp_at(temperatures, sndstamp_unix, target_unix)`** – linearly interpolates the temperature array at the fractional grid index corresponding to `target_unix`, exactly replicating the `WeatherForecastModel.predict_temperature_at_time` method.

In [ ]:
def sun_altitude_deg(t: Time) -> float:
    sun = get_sun(t)
    return sun.transform_to(AltAz(obstime=t, location=RUBIN, pressure=0)).alt.deg


def next_end_of_evening_twilight(
    after: Time, target_alt: float = NAUTICAL_TWILIGHT_DEG
) -> Time:
    """Next time the sun crosses `target_alt` while descending, after `after`."""
    t0 = after + 1 * u.s
    alt_before = sun_altitude_deg(t0)
    for i in range(1, 26):
        t1 = t0 + i * u.hour
        alt_after = sun_altitude_deg(t1)
        if alt_before > target_alt and alt_after < target_alt:
            break
        alt_before = alt_after
    else:
        raise RuntimeError("No descending crossing found in next 25 hours.")

    def f(t_sec: float) -> float:
        return sun_altitude_deg(Time(t_sec, format="unix")) - target_alt

    return Time(brentq(f, (t1 - 1 * u.hour).unix, t1.unix), format="unix")


def interp_at(temperatures: np.ndarray, sndstamp_unix: float, target_unix: float):
    """Linear interpolation matching WeatherForecastModel.predict_temperature_at_time."""
    index_float = (target_unix - sndstamp_unix) / DELTA_TIME - 1
    if index_float < 0 or index_float > len(temperatures) - 1:
        return np.nan
    return float(np.interp(index_float, np.arange(len(temperatures)), temperatures))

## Compute Predicted Twilight Temperatures

For every forecast issuance in the dataset:
1. Compute the next end-of-evening nautical twilight after the message's send timestamp.
2. Interpolate the forecast's temperature grid at that twilight moment.

Results are assembled into a tidy DataFrame with columns for publication time, the predicted twilight time, and the predicted temperature.

In [ ]:
snd = df["private_sndStamp"].to_numpy(dtype=float)
temps = df[temperature_fields].to_numpy(dtype=float)

predictions = np.empty(len(df))
twilights = np.empty(len(df))
for i in range(len(df)):
    twilights[i] = next_end_of_evening_twilight(
        Time(float(snd[i]), format="unix")
    ).tai.unix
    predictions[i] = interp_at(temps[i], float(snd[i]), twilights[i])

result = pd.DataFrame(
    {
        "publication_time": pd.to_datetime(snd, unit="s", utc=True),
        "twilight_time": pd.to_datetime(twilights, unit="s", utc=True),
        "predicted_twilight_temperature": predictions,
    }
).sort_values("publication_time")
result.head()

## Fetch Actual ESS:301 Temperature

Query the actual outside air temperature from ESS sensor index 301 over the full time span covered by the forecasts (from the earliest publication time through the end of the forecast horizon). This provides ground truth to compare against the predictions.

In [ ]:
ESS_TOPIC = "lsst.sal.ESS.temperature"
ESS_INDEX = 301

# Query from earliest forecast publication through the end of the longest forecast horizon
ess_start = Time(snd.min(), format="unix")
ess_end = Time(snd.max() + len(temperature_fields) * DELTA_TIME, format="unix")

ess_df = await client.select_time_series(
    ESS_TOPIC,
    ["temperatureItem0", "sensorName", "salIndex"],
    ess_start,
    ess_end,
    index=ESS_INDEX,
)
print(f"Fetched {len(ess_df)} ESS:301 temperature samples.")

# Extract actual temperature at twilight via interpolation
ess_times_unix = ess_df.index.to_series().apply(lambda t: t.timestamp()).values
ess_temps_vals = ess_df["temperatureItem0"].values.astype(float)
twi_unix = twilights[0]
actual_twi_temp = float(np.interp(twi_unix, ess_times_unix, ess_temps_vals))
print(f"Actual ESS:301 temperature at twilight: {actual_twi_temp:.2f} °C")

## Visualisation

Plot the predicted twilight temperature as a function of forecast publication time. Each point represents one `hourlyTrend` issuance; the trend shows how the model's estimate of tonight's opening temperature evolved as successive forecasts were published throughout the day.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
valid = result.dropna(subset=["predicted_twilight_temperature"])

ax.plot(
    valid["publication_time"],
    valid["predicted_twilight_temperature"],
    marker=".",
    linestyle="-",
    label="Forecast prediction",
)

ax.axhline(
    actual_twi_temp,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"Actual ESS:301 at twilight ({actual_twi_temp:.1f} °C)",
)

ax.set_xlabel("Publication time (UTC)")
ax.set_ylabel("Temperature at twilight (°C)")
ax.set_title("WeatherForecast.hourlyTrend: predicted vs. actual twilight temperature")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Lead Time vs. Predicted Temperature

How early does the forecast converge? Each point shows the predicted twilight temperature plotted against how many hours before twilight the forecast was issued. If the forecast is well-behaved, points should stabilize (flatten) as lead time decreases toward zero.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
valid = result.dropna(subset=["predicted_twilight_temperature"])

lead_time_hours = (
    valid["twilight_time"] - valid["publication_time"]
).dt.total_seconds() / 3600.0

ax.plot(
    lead_time_hours,
    valid["predicted_twilight_temperature"],
    marker="o",
    linestyle="-",
    markersize=5,
    label="Forecast prediction",
)

ax.axhline(
    actual_twi_temp,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"Actual ESS:301 at twilight ({actual_twi_temp:.1f} °C)",
)

ax.set_xlabel("Lead time before twilight (hours)")
ax.set_ylabel("Temperature at twilight (°C)")
ax.set_title("Forecast convergence: predicted vs. actual twilight temperature")
ax.invert_xaxis()
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Full Forecast Temperature Curves

Each faint line is the full temperature prediction curve from one forecast issuance, plotted in absolute UTC time. The coloured dot marks where that curve is sampled to extract the twilight temperature. This reveals the shape of the predicted cooling ramp and highlights any data gaps or inflection points near the extraction moment.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

cmap = plt.cm.viridis
norm = plt.Normalize(0, len(df) - 1)

for i in range(len(df)):
    row_temps = temps[i]
    valid_mask = ~np.isnan(row_temps)
    n_valid = valid_mask.sum()
    if n_valid == 0:
        continue

    forecast_times = pd.to_datetime(
        snd[i] + (np.arange(n_valid) + 1) * DELTA_TIME, unit="s", utc=True
    )
    ax.plot(
        forecast_times,
        row_temps[valid_mask],
        color=cmap(norm(i)),
        alpha=0.35,
        linewidth=0.8,
    )

    twi_time = pd.to_datetime(twilights[i], unit="s", utc=True)
    pred_temp = predictions[i]
    if not np.isnan(pred_temp):
        ax.plot(twi_time, pred_temp, "o", color=cmap(norm(i)), markersize=5, zorder=5)

# Actual ESS:301 temperature: split into "fitted" (before last forecast) and "non-fitted" (after)
last_pub_time = pd.to_datetime(snd.max(), unit="s", utc=True)
ess_times_dt = ess_df.index

fitted_mask = ess_times_dt <= last_pub_time
nonfitted_mask = ess_times_dt > last_pub_time

ax.plot(
    ess_times_dt[fitted_mask],
    ess_temps_vals[fitted_mask],
    color="black",
    linewidth=1.8,
    label="Actual ESS:301 (fitted region)",
    zorder=10,
)
ax.plot(
    ess_times_dt[nonfitted_mask],
    ess_temps_vals[nonfitted_mask],
    color="red",
    linewidth=1.8,
    label="Actual ESS:301 (non-fitted / future)",
    zorder=10,
)

twi_time_common = pd.to_datetime(twilights[0], unit="s", utc=True)
ax.axvline(
    twi_time_common, color="red", linestyle="--", alpha=0.6, label="Nautical twilight"
)

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Forecast curves vs. actual ESS:301 temperature")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()